# Agentic RAG — Pipeline Demo

## Setup

In [1]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import main as main_module
importlib.reload(main_module)

from main import agentic_rag_app
print("Pipeline loaded.")

c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6756.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline loaded.


## Helper — Stream Runner

In [2]:
def run_demo(query: str, namespace: str = "default_namespace"):
    """Stream the agentic RAG graph and print a clean per-node trace."""
    initial_state = {
        "query": query,
        "original_query": query,
        "namespace": namespace,
        "documents": [],
        "final_context_strips": [],
        "needs_web_search": False,
        "answer": "",
        "retries": 0,
        "hallucination_retries": 0,
        "route": ""
    }

    print(f"Query : {query}")
    print("=" * 65)

    nodes_visited = []

    for chunk in agentic_rag_app.stream(initial_state):
        for node, update in chunk.items():
            nodes_visited.append(node)
            print(f"\n---> Node [{node}]")

            if "route" in update and update["route"]:
                print(f"  Route              : {update['route']}")
            if "documents" in update:
                print(f"  Docs fetched       : {len(update['documents'])}")
            if "final_context_strips" in update:
                strips = update["final_context_strips"]
                print(f"  Context strips     : {len(strips)}")
            if "needs_web_search" in update:
                print(f"  Needs rewrite/web  : {update['needs_web_search']}")
            if "retries" in update:
                print(f"  Retries            : {update['retries']}")
            if "hallucination_retries" in update:
                print(f"  Halluc retries     : {update['hallucination_retries']}")
            if "answer" in update and update["answer"]:
                print(f"\n  Answer:\n  {update['answer']}")

    print("\n" + "=" * 65)
    print(f"  Node path: {' → '.join(nodes_visited)}")
    print("=" * 65)

---
## LLM Direct

**When triggered:** The Adaptive Router classifies a query as general knowledge that the LLM can answer from its parametric memory — no retrieval needed.

In [3]:
# General-knowledge question — should be classified as llm_direct
run_demo("Explain the concept of entropy in thermodynamics in simple terms.")

Query : Explain the concept of entropy in thermodynamics in simple terms.
2026-03-29 17:31:57 - src.adaptive_router - INFO - Query 'Explain the concept of entropy in thermodynamics in simple terms.' classified to route: 'llm_direct'

---> Node [router]
  Route              : llm_direct

---> Node [generate]

  Answer:
  In simple terms, entropy is a measure of disorder or randomness in a system. Imagine you have a tidy room where everything is organized and in its place. This is like a state of low entropy. Now, imagine the same room after a big party, with clothes scattered all over the floor, furniture out of place, and trash everywhere. This is like a state of high entropy.

In thermodynamics, entropy works in a similar way. It measures how much energy is unavailable to do work because it's dispersed or wasted. When energy is concentrated and organized, like in a tidy room, it's available to do work. But when energy is dispersed and random, like in a messy room, it's less useful.

F

In [4]:
# Another example — open-ended reasoning, no document context needed
run_demo("What are the main differences between supervised and unsupervised machine learning?")

Query : What are the main differences between supervised and unsupervised machine learning?
2026-03-29 17:32:01 - src.adaptive_router - INFO - Query 'What are the main differences between supervised and unsupervised machine learning?' classified to route: 'llm_direct'

---> Node [router]
  Route              : llm_direct

---> Node [generate]

  Answer:
  The main differences between supervised and unsupervised machine learning are:

1. **Presence of labeled data**: In supervised learning, the machine learning algorithm is trained on labeled data, meaning the data is already categorized or classified. In unsupervised learning, the algorithm is trained on unlabeled data, and it must find patterns or structure in the data on its own.

2. **Learning objective**: The primary objective of supervised learning is to learn a mapping between input data and the corresponding output labels, so the algorithm can make predictions on new, unseen data. In contrast, the goal of unsupervised learning i

---
## Web Search (DuckDuckGo)

**When triggered:** Real-time, current-events, or live-data queries that require information beyond the LLM's training cutoff.

**Node path:** `router → web_search → generate`

**Key behaviours:**
- DuckDuckGo search result is appended as a single context strip with prefix `[Web Search Results]:`
- `hallucination_grader` is **bypassed** — web results are treated as ground truth (`route != 'vectorstore'` short-circuits the check)
- `answer_quality_grader` **does** run — if the answer is off-topic, `rewrite_query` fires and routes back to `web_search` (not vectorstore)
- Hard retry cap: `retries >= 3` → `END` unconditionally

In [5]:
# Real-time query — should be classified as web_search
run_demo("What is the current stock price of NVIDIA (NVDA) and the latest news?")

Query : What is the current stock price of NVIDIA (NVDA) and the latest news?
2026-03-29 17:32:15 - src.adaptive_router - INFO - Query 'What is the current stock price of NVIDIA (NVDA) and the latest news?' classified to route: 'web_search'

---> Node [router]
  Route              : web_search

---> Node [web_search]
  Context strips     : 1
2026-03-29 17:32:17 - src.workflows.self_rag - INFO - Answer Quality Score: no

---> Node [generate]

  Answer:
  The provided context does not give the current stock price of NVIDIA (NVDA) or the latest news. However, it suggests that you can find this information at websites such as MarketBeat, NVIDIA stock page, or Barron's, which provide real-time stock prices, news, and analysis. To get the current stock price and latest news, you would need to visit one of these websites.
2026-03-29 17:32:18 - src.workflows.self_rag - INFO - Query Rewritten: 'What is the current stock price of NVIDIA (NVDA) and the latest news?' -> 'To better address the inqu

In [6]:
# Another real-time example — current events
run_demo("What are the latest developments in AI regulation in the EU in 2025?")

Query : What are the latest developments in AI regulation in the EU in 2025?
2026-03-29 17:32:39 - src.adaptive_router - INFO - Query 'What are the latest developments in AI regulation in the EU in 2025?' classified to route: 'web_search'

---> Node [router]
  Route              : web_search

---> Node [web_search]
  Context strips     : 1
2026-03-29 17:32:41 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  The latest developments in AI regulation in the EU in 2025 include:

1. **Introduction of the EU AI Act**: The EU AI Act introduces transformative regulatory obligations for General-Purpose AI model providers, with enforcement beginning in August 2025.
2. **Advisory bodies and conformity assessments**: Companies are responsible for contacting advisory bodies (opening on August 2, 2025) to understand their obligations, and from August 2, 2026, they will need to have a conformity assessment carried out by notified bodies.
3. **Registration

---
## Vectorstore

**When triggered:** Document-specific queries where the answer is expected to live in the indexed knowledge base (Pinecone + MongoDB).

**Node path (happy path that we want):** `router → retrieve → evaluate → generate`

**Node path (zero helpful strips):** `router → retrieve → evaluate → rewrite_query → retrieve → evaluate → generate`

**Key behaviours:**

| Stage | What happens |
|---|---|
| **Retrieve** | VectorDB search (+ optional Pinecone Reranker). `parent_id` → MongoDB fetch for full parent chunks |
| **CRAG Evaluate** | Each doc rated `correct / ambiguous / incorrect`. Strips extracted from `correct` and `ambiguous` docs |
| **Zero-strip guard** | `needs_web_search = True` only if `len(final_context_strips) == 0` after ALL docs graded |
| **Self-RAG: Hallucination** | LLM checks answer is grounded in retrieved strips (`yes` / `no`) |
| **Self-RAG: Quality** | LLM checks answer resolves the original query (`yes` / `no`) |
| **Retry hard cap** | `retries >= 3` or `hallucination_retries >= 3` → `END` immediately |

In [7]:
# Document-specific query about the indexed knowledge base (RAG-Anything paper)
run_demo("How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.")

Query : How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.
2026-03-29 17:32:58 - src.adaptive_router - INFO - Query 'How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.' classified to route: 'vectorstore'

---> Node [router]
  Route              : vectorstore
2026-03-29 17:32:58 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-29 17:33:00 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)

---> Node [retrieve]
  Docs fetched       : 5
2026-03-29 17:33:33 - src.workflows.crag - INFO - CRAG evaluate: 29 relevant strips found. needs_web_search=False

---> Node [evaluate]
  Context strips     : 29
  Needs rewrite/web  : False
2026-03-29 17:33:39 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-29 17:33:41 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  The hybrid retrieval engine in RAG-Anything combines two pat

In [8]:
# Another vectorstore query
run_demo("What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?")

Query : What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?
2026-03-29 17:33:50 - src.adaptive_router - INFO - Query 'What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?' classified to route: 'vectorstore'

---> Node [router]
  Route              : vectorstore
2026-03-29 17:33:50 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-29 17:33:51 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)

---> Node [retrieve]
  Docs fetched       : 5
2026-03-29 17:35:35 - src.workflows.crag - INFO - CRAG evaluate: 21 relevant strips found. needs_web_search=False

---> Node [evaluate]
  Context strips     : 21
  Needs rewrite/web  : False
2026-03-29 17:35:40 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-29 17:35:42 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  The dual-graph architecture in RAG-Anything consists of tw